In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer

In [15]:
df = pd.read_csv('/content/healthcare-dataset-stroke-data.csv')

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [17]:
df.describe()


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [18]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [19]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0
5109,44679,Female,44.0,0,0,Yes,Govt_job,Urban,85.28,26.2,Unknown,0


In [20]:
target_col = 'Residence_type'
col_idx = df.columns.get_loc(target_col)


In [22]:
#One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False)
encoded_data = encoder.fit_transform(df[[target_col]])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out([target_col]))

In [23]:

df_no_residence = df.drop(columns=[target_col])
df = pd.concat([df_no_residence.iloc[:, :col_idx], encoded_df, df_no_residence.iloc[:, col_idx:]], axis=1)

df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type_Rural,Residence_type_Urban,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,0.0,1.0,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,1.0,0.0,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,1.0,0.0,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,0.0,1.0,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,1.0,0.0,174.12,24.0,never smoked,1


In [25]:
missing_counts = df.isnull().sum()
missing_percents = (df.isnull().sum() / len(df)) * 100

analysis = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percents
})
print(analysis[analysis['Missing Count'] > 0])

     Missing Count  Percentage (%)
bmi            201        3.933464


In [26]:
for col in df.columns:
    if df[col].isnull().any():
        percent = (df[col].isnull().sum() / len(df)) * 100

        if percent < 20:
            print(f"Applying SimpleImputer (Mean) on {col} because missing is {percent:.2f}%")
            imputer = SimpleImputer(strategy='mean')
        else:
            print(f"Applying KNNImputer on {col} because missing is {percent:.2f}%")
            imputer = KNNImputer(n_neighbors=5)

        df[col] = imputer.fit_transform(df[[col]])


print("\nMissing values after imputation:", df.isnull().sum().sum())
df.head()

Applying SimpleImputer (Mean) on bmi because missing is 3.93%

Missing values after imputation: 0


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type_Rural,Residence_type_Urban,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,0.0,1.0,228.69,36.600000,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,1.0,0.0,202.21,28.893237,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,1.0,0.0,105.92,32.500000,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,0.0,1.0,171.23,34.400000,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,1.0,0.0,174.12,24.000000,never smoked,1
